# LangChain RAG 逐步复现
## 使用方式

1. 建议先在当前目录打开这个 Notebook。
2. 优先运行前面的“环境检查”单元。
3. 如果没有配置 `DASHSCOPE_API_KEY` 或没有启动 `Ollama`，相关章节会自动跳过并提示原因。
4. 可以按顺序逐段学习，也可以直接点击“全部运行”。

## 推荐依赖安装命令

```bash
python -m pip install -U langchain langchain-core langchain-community langchain-text-splitters langchain-ollama langchain-chroma chromadb dashscope jq pypdf
```


In [ ]:
#直接在notebook中安装依赖环境
%pip install -U langchain langchain-core langchain-community langchain-text-splitters langchain-ollama langchain-chroma chromadb dashscope jq pypdf

## 00. 环境检查与公共配置

- 这一节不对应单独的 `.py` 文件，专门用来让后面的案例更稳地执行。
- 会检查当前目录、数据目录、`DASHSCOPE_API_KEY`、`Ollama` 服务端口和若干可选依赖。
- 后续章节如果前置条件不足，会显示“跳过原因”而不是直接报错。


In [2]:
from __future__ import annotations

import importlib.util
import os
import socket
from pathlib import Path


def locate_project_root() -> Path:
    signatures = [("data/stu.csv", "data/info.csv", "data/Python基础语法.txt")]
    seen: set[Path] = set()
    candidates: list[Path] = []

    def add_candidate(path: Path) -> None:
        resolved = path.resolve()
        if resolved not in seen and resolved.exists() and resolved.is_dir():
            seen.add(resolved)
            candidates.append(resolved)

    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        add_candidate(base)
        try:
            child_dirs = [child for child in base.iterdir() if child.is_dir()]
        except PermissionError:
            child_dirs = []
        for child in child_dirs:
            add_candidate(child)
            try:
                grandchild_dirs = [grandchild for grandchild in child.iterdir() if grandchild.is_dir()]
            except PermissionError:
                grandchild_dirs = []
            for grandchild in grandchild_dirs:
                add_candidate(grandchild)

    for candidate in candidates:
        if all((candidate / rel).exists() for rel in signatures[0]):
            return candidate

    raise FileNotFoundError("没有找到 02_LangChainRAG开发 目录，请确认从仓库目录中打开 Notebook。")


ROOT_DIR = locate_project_root()
os.chdir(ROOT_DIR)
DATA_DIR = ROOT_DIR / "data"
CHAT_HISTORY_DIR = ROOT_DIR / "chat_history"
CHROMA_DIR = ROOT_DIR / "chroma_db"
CHAT_HISTORY_DIR.mkdir(exist_ok=True)
CHROMA_DIR.mkdir(exist_ok=True)


def is_port_open(host: str = "127.0.0.1", port: int = 11434, timeout: float = 1.0) -> bool:
    sock = socket.socket()
    sock.settimeout(timeout)
    try:
        sock.connect((host, port))
        return True
    except OSError:
        return False
    finally:
        sock.close()


HAS_DASHSCOPE = bool(os.getenv("DASHSCOPE_API_KEY"))
HAS_OLLAMA = is_port_open()
HAS_JQ = importlib.util.find_spec("jq") is not None
HAS_PYPDF = importlib.util.find_spec("pypdf") is not None


def section_ready(section_id: str, condition: bool, reason: str) -> bool:
    if condition:
        print(f"[运行] {section_id}")
        return True
    print(f"[跳过] {section_id}: {reason}")
    return False


print(f"ROOT_DIR = {ROOT_DIR}")
print(f"DATA_DIR = {DATA_DIR}")
print(f"DASHSCOPE_API_KEY = {'已配置' if HAS_DASHSCOPE else '未配置'}")
print(f"Ollama(11434) = {'可连接' if HAS_OLLAMA else '不可连接'}")
print(f"jq = {'已安装' if HAS_JQ else '未安装'}")
print(f"pypdf = {'已安装' if HAS_PYPDF else '未安装'}")


ROOT_DIR = d:\Agent\03-RAG\class\代码\AI大模型RAG与智能体开发\P3_LangChainRAG开发
DATA_DIR = d:\Agent\03-RAG\class\代码\AI大模型RAG与智能体开发\P3_LangChainRAG开发\data
DASHSCOPE_API_KEY = 已配置
Ollama(11434) = 可连接
jq = 已安装
pypdf = 已安装


## 01. 01[扩展]余弦相似度

- 对应源码：`01[扩展]余弦相似度.py`
- 本节说明：用最基础的方式理解向量点积、模长与余弦相似度，为后面的嵌入与检索打底。
- 运行提示：直接可运行，无需额外前置条件。


In [3]:
import numpy as np

"""
计算两个向量的余弦相似度（衡量方向相似性，剔除长度影响）
"""


def get_dot(vec_a, vec_b):
    """计算 2 个向量的点积。"""
    if len(vec_a) != len(vec_b):
        raise ValueError("2个向量必须维度数量相同")

    dot_sum = 0
    for a, b in zip(vec_a, vec_b):
        dot_sum += a * b

    return dot_sum


def get_norm(vec):
    """计算单个向量的模长。"""
    sum_square = 0
    for v in vec:
        sum_square += v * v

    return np.sqrt(sum_square)


def cosine_similarity(vec_a, vec_b):
    """余弦相似度：点积 / 模长乘积。"""
    return get_dot(vec_a, vec_b) / (get_norm(vec_a) * get_norm(vec_b))


if __name__ == "__main__":
    vec_a = [0.5, 0.5]
    vec_b = [0.7, 0.7]
    vec_c = [0.7, 0.5]
    vec_d = [-0.6, -0.5]

    print("ab:", cosine_similarity(vec_a, vec_b))
    print("ac:", cosine_similarity(vec_a, vec_c))
    print("ad:", cosine_similarity(vec_a, vec_d))


ab: 1.0
ac: 0.9863939238321436
ad: -0.995893206467704


## 02. 02LangChain访问阿里云通义千问大模型

- 对应源码：`02LangChain访问阿里云通义千问大模型.py`
- 本节说明：演示如何通过 LangChain 调用阿里云通义千问文本模型，这一节需要 `DASHSCOPE_API_KEY`。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [4]:
if section_ready("02", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.llms import Tongyi

    # qwen-max 适合作为文本补全模型使用
    model = Tongyi(model="qwen-max")

    res = model.invoke("你是谁呀，能做什么？")
    print(res)


[运行] 02
您好，我是Qwen，全名通义千问，是阿里云自主研发的超大规模语言模型。我能够帮助您回答问题、创作文字，比如写故事、写公文、写邮件、写剧本等等，还能表达观点，玩游戏等。如果您有任何问题或需要帮助，请随时告诉我，我会尽力提供支持。


## 03. 03LangChain访问Ollama本地模型

- 对应源码：`03LangChain访问Ollama本地模型.py`
- 本节说明：演示如何连接本地 Ollama 大模型，这一节要求本机 `Ollama` 已启动并已拉取对应模型。
- 运行提示：运行前置：未检测到本地 Ollama 服务（默认端口 11434）。若条件不满足，本节会自动跳过。


In [5]:
if section_ready("03", HAS_OLLAMA, "未检测到本地 Ollama 服务（默认端口 11434）"):
    from langchain_ollama import OllamaLLM

    model = OllamaLLM(model="qwen3:4b")

    res = model.invoke("你是谁呀，能做什么？")
    print(res)


[运行] 03
你好！我是**通义千问**（Qwen），是阿里巴巴集团旗下的通义实验室自主研发的超大规模语言模型。我可以帮助你：

1. **回答问题**：无论是学术问题、生活常识，还是专业领域的问题，我都可以尝试为你解答。
2. **创作文字**：写故事、公文、邮件、剧本等，甚至帮你写诗、写歌词。
3. **编程**：理解并生成多种编程语言的代码，比如Python、Java、C++等。
4. **逻辑推理**：解决数学问题、逻辑谜题，或者进行数据分析。
5. **多语言翻译**：支持中、英、德、法、西班牙等百种语言的翻译。
6. **表达观点**：针对热点话题或复杂问题，提供有深度的分析和建议。

比如，你可以让我帮你写一封英文邮件、用Python写一个简单的程序，或者解释量子力学的基本概念。  
**你有什么需要我帮忙的吗？** 😊


## 04. 04LangChain的流式输出

- 对应源码：`04LangChain的流式输出.py`
- 本节说明：对比云端模型和本地模型的流式输出效果，便于理解 `stream()` 的用法。
- 运行提示：直接可运行，无需额外前置条件。


In [6]:
print("## 通义千问流式输出")
if section_ready("04-A", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.llms import Tongyi

    model = Tongyi(model="qwen-max")
    for chunk in model.stream("你是谁呀，能做什么？"):
        print(chunk, end="", flush=True)
    print("\n")

print("## Ollama 流式输出")
if section_ready("04-B", HAS_OLLAMA, "未检测到本地 Ollama 服务（默认端口 11434）"):
    from langchain_ollama import OllamaLLM

    model = OllamaLLM(model="qwen3:4b")
    for chunk in model.stream("你是谁呀，能做什么？"):
        print(chunk, end="", flush=True)
    print()


## 通义千问流式输出
[运行] 04-A
你好！我是Qwen，是阿里云开发的一款超大规模语言模型。我被设计用来生成各种类型的文本，如文章、故事、诗歌、故事等，并能够根据不同的场景和需求进行变换和扩展。此外，我还能够帮助用户完成多种任务，比如回答问题、提供建议、撰写邮件或报告、辅助学习新知识等。简而言之，无论你需要创意写作还是寻求信息解答，我都会尽力提供支持。有什么我可以帮到你的吗？

## Ollama 流式输出
[运行] 04-B
你好！我是**通义千问**（Qwen），阿里巴巴集团旗下的通义实验室自主研发的超大规模语言模型。你可以叫我**通义**或者**Qwen**哦～

### 🌟 我能帮你做什么？
1. **回答问题**  
   - 学习、生活、科技、文化、历史……几乎所有领域的问题都能解答。
   - 例如：*“如何用Python写一个简单的计算器？”*、*“唐朝的诗人李白和杜甫有什么区别？”*

2. **创作文字**  
   - 写故事、写公文、写邮件、写剧本、写诗、写歌词……  
   - 例如：*“请写一个关于未来城市的科幻短篇故事”*、*“帮我写一封给老板的请假邮件”*

3. **逻辑推理与编程**  
   - 解题、写代码、调试程序、数学计算……  
   - 例如：*“求解这个方程：x² + 2x - 3 = 0”*、*“用Python实现一个斐波那契数列生成器”*

4. **表达观点**  
   - 分析问题、给出建议、讨论热点话题……  
   - 例如：*“人工智能对社会的影响是什么？”*、*“如何提高学习效率？”*

5. **多语言支持**  
   - 除了中文，我还支持英文、德语、法语、西班牙语等百种语言！  
   - 例如：*“用英语写一首关于爱的诗”*

---

### 💡 小例子：
- 你问：*“今天天气怎么样？”* → 我可以帮你查天气（如果联网的话）  
- 你问：*“帮我写一个生日祝福语”* → 我直接生成一段温馨的文字  
- 你问：*“为什么天空是蓝色的？”* → 我会用简单易懂的语言解释光的散射原理

---

**如果你有任何问题或需要帮助，随时告诉我！** 😊  
比如：  
- 需要写一段代码？  
- 想了解某个知识点？  
- 或者……想和我玩个游戏？  

我随时在！ ✨


## 05. 05LangChain调用聊天模型

- 对应源码：`05LangChain调用聊天模型.py`
- 本节说明：演示消息列表形式的聊天模型调用，重点观察 `System / Human / AI` 三类消息如何协作。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [7]:
if section_ready("05", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

    model = ChatTongyi(model="qwen3-max")

    messages = [
        SystemMessage(content="你是一个边塞诗人。"),
        HumanMessage(content="写一首唐诗。"),
        AIMessage(content="锄禾日当午，汗滴禾下土，谁知盘中餐，粒粒皆辛苦。"),
        HumanMessage(content="按照你上一个回复的格式，再写一首唐诗。"),
    ]

    for chunk in model.stream(messages):
        print(chunk.content, end="", flush=True)


[运行] 05
烽火照边城，铁衣凝夜霜。  
黄沙埋战骨，孤雁唳寒荒。  
月冷弓刀影，风悲鼓角声。  
何须问归处，马革裹尸还。

## 06. 06LangChain调用Ollama的聊天模型

- 对应源码：`06LangChain调用Ollama的聊天模型.py`
- 本节说明：把上一节的聊天模型调用方式切换到本地 Ollama 版本。
- 运行提示：运行前置：未检测到本地 Ollama 服务（默认端口 11434）。若条件不满足，本节会自动跳过。


In [8]:
if section_ready("06", HAS_OLLAMA, "未检测到本地 Ollama 服务（默认端口 11434）"):
    from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
    from langchain_ollama import ChatOllama

    model = ChatOllama(model="qwen3:4b")

    messages = [
        SystemMessage(content="你是一个边塞诗人。"),
        HumanMessage(content="写一首唐诗。"),
        AIMessage(content="锄禾日当午，汗滴禾下土，谁知盘中餐，粒粒皆辛苦。"),
        HumanMessage(content="按照你上一个回复的格式，再写一首唐诗。"),
    ]

    for chunk in model.stream(messages):
        print(chunk.content, end="", flush=True)


[运行] 06
烽烟连塞垣，铁马踏霜寒。将士思家切，月明照孤鞍。

## 07. 07LangChain消息的简写形式

- 对应源码：`07LangChain消息的简写形式.py`
- 本节说明：说明 LangChain 中消息也可以用元组简写，写法更紧凑。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [9]:
if section_ready("07", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi

    model = ChatTongyi(model="qwen3-max")

    messages = [
        ("system", "你是一个边塞诗人。"),
        ("human", "写一首唐诗。"),
        ("ai", "锄禾日当午，汗滴禾下土，谁知盘中餐，粒粒皆辛苦。"),
        ("human", "按照你上一个回复的格式，再写一首唐诗。"),
    ]

    for chunk in model.stream(messages):
        print(chunk.content, end="", flush=True)


[运行] 07
烽火照边城，霜刃映寒星。  
黄沙埋战骨，孤雁唳空营。  
月冷弓刀湿，风高鼓角鸣。  
何须封侯印，但愿海波平。

## 08. 08LangChain访问阿里云嵌入模型

- 对应源码：`08LangChain访问阿里云嵌入模型.py`
- 本节说明：演示查询向量与批量文档向量的生成，后面做向量检索都会依赖这个能力。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [10]:
if section_ready("08", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.embeddings import DashScopeEmbeddings

    model = DashScopeEmbeddings(model="text-embedding-v4")

    print(model.embed_query("我喜欢你"))
    print(model.embed_documents(["我喜欢你", "我稀饭你", "晚上吃啥"]))


[运行] 08
[-0.008780702017247677, -0.07309616357088089, 0.02290618047118187, -0.020386500284075737, 0.0638318881392479, -0.06912576407194138, 0.025604018941521645, 0.06510445475578308, -0.05833440274000168, 0.13356848061084747, 0.02084462344646454, 0.03547912836074829, 0.03458833321928978, -0.03838057816028595, -0.003202093066647649, -0.020615560933947563, -0.021200941875576973, 0.038024257868528366, -0.0003706347197294235, -0.0007050808635540307, -0.055229343473911285, -0.008316216059029102, 0.06220300495624542, -0.010396860539913177, -0.05161525681614876, -0.02037377469241619, 0.010135984979569912, -0.037438876926898956, -0.06795500218868256, 0.02756376937031746, -0.030439767986536026, -0.00816987082362175, -0.04578690975904465, 0.016810590401291847, -0.043852608650922775, 0.0031655067577958107, 0.002222217619419098, 0.02500591240823269, -0.04125657677650452, 0.0215827114880085, 0.019152112305164337, -0.035682737827301025, 0.008901596069335938, 0.02299525961279869, 0.04090025648474693,

## 09. 09LangChain访问Ollama的本地嵌入模型

- 对应源码：`09LangChain访问Ollama的本地嵌入模型.py`
- 本节说明：将嵌入模型切换为本地 Ollama 版本，便于离线场景实验。
- 运行提示：运行前置：未检测到本地 Ollama 服务（默认端口 11434）。若条件不满足，本节会自动跳过。


In [11]:
if section_ready("09", HAS_OLLAMA, "未检测到本地 Ollama 服务（默认端口 11434）"):
    from langchain_ollama import OllamaEmbeddings

    model = OllamaEmbeddings(model="qwen3-embedding:4b")

    print(model.embed_query("我喜欢你"))
    print(model.embed_documents(["我喜欢你", "我稀饭你", "晚上吃啥"]))


[运行] 09
[-0.00025322076, 0.01648288, 0.047239058, 0.012868762, -0.0017447375, -0.01065702, 0.080076165, -0.018544767, 0.025343502, 0.010887394, 0.049638975, -0.013264701, -0.00024263914, -0.02501076, -0.028675668, -0.011536011, -0.0055275555, -0.02140573, 0.0012317874, -0.004687489, -0.0032302074, -0.0029158914, 0.018946506, -0.025595875, -0.0061263903, -0.0056544985, -0.0176414, -0.007277243, 0.048730005, -0.030701613, -0.0453673, -0.05827569, 0.0044598747, 0.0036095404, 0.0037171661, 0.013357589, -0.04408367, -0.0048631756, -0.011521484, -0.0073500997, 0.022300987, -0.031871993, 0.027014341, 0.0068935114, -0.024436653, -0.04177566, 0.003544306, 0.019313311, 0.0008131712, -0.007108725, -0.0016582347, 0.019334486, 0.011376452, -0.011422159, -0.0045772376, 0.008833494, 0.034636285, -0.015798962, -0.034892812, -0.007146012, 0.012718747, 0.019408591, -0.028259847, 0.009903137, -0.0053291116, -0.024171455, -0.028165314, -0.030862607, 0.0071832784, -0.017309822, -0.02000378, -0.020096375, 0

## 10. 10通用提示词模板

- 对应源码：`10通用提示词模板.py`
- 本节说明：先从最常见的 `PromptTemplate` 入手，体会变量填充和链式调用。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [12]:
if section_ready("10", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.llms import Tongyi
    from langchain_core.prompts import PromptTemplate

    prompt_template = PromptTemplate.from_template(
        "我的邻居姓{lastname}，刚生了{gender}，你帮我起个名字，简单回答。"
    )

    model = Tongyi(model="qwen-max")
    chain = prompt_template | model

    res = chain.invoke({"lastname": "张", "gender": "女儿"})
    print(res)


[运行] 10
可以考虑“张婉儿”这个名字，既优雅又易于记忆。


## 11. 11FewShot提示词模板

- 对应源码：`11FewShot提示词模板.py`
- 本节说明：通过给示例的方式约束模型输出，观察 Few-shot Prompt 的组织结构。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [13]:
if section_ready("11", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.llms import Tongyi
    from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

    example_template = PromptTemplate.from_template("单词：{word}，反义词：{antonym}")

    examples_data = [
        {"word": "大", "antonym": "小"},
        {"word": "上", "antonym": "下"},
    ]

    few_shot_template = FewShotPromptTemplate(
        example_prompt=example_template,
        examples=examples_data,
        prefix="告知我单词的反义词，我提供如下的示例：",
        suffix="基于前面的示例告知我，{input_word} 的反义词是？",
        input_variables=["input_word"],
    )

    prompt_text = few_shot_template.format(input_word="左")
    print(prompt_text)

    model = Tongyi(model="qwen-max")
    print(model.invoke(prompt_text))


[运行] 11
告知我单词的反义词，我提供如下的示例：

单词：大，反义词：小

单词：上，反义词：下

基于前面的示例告知我，左 的反义词是？
基于您给出的示例，“左”的反义词是“右”。


## 12. 12模板类的format和invoke方法

- 对应源码：`12模板类的format和invoke方法.py`
- 本节说明：区分模板的 `format()` 与 `invoke()`，理解字符串模板和消息模板的返回值差异。
- 运行提示：直接可运行，无需额外前置条件。


In [14]:
from langchain_core.prompts import ChatPromptTemplate, FewShotPromptTemplate, PromptTemplate

template = PromptTemplate.from_template("我的邻居是：{lastname}，最喜欢：{hobby}")

res = template.format(lastname="张大明", hobby="钓鱼")
print(res, type(res))

res2 = template.invoke({"lastname": "周杰轮", "hobby": "唱歌"})
print(res2, type(res2))


few_shot_template = FewShotPromptTemplate(
    example_prompt=PromptTemplate.from_template("问题：{question}，答案：{answer}"),
    examples=[
        {"question": "1+1等于几？", "answer": "2"},
        {"question": "中国的首都是哪？", "answer": "北京"},
    ],
    prefix="参考下面示例：",
    suffix="问题：{input_question}，答案：",
    input_variables=["input_question"],
)

res3 = few_shot_template.format(input_question="太阳从哪边升起？")
print(res3, type(res3))


chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个 helpful assistant。"),
        ("human", "{question}"),
    ]
)

res4 = chat_template.invoke({"question": "你是谁？"})
print(res4, type(res4))


我的邻居是：张大明，最喜欢：钓鱼 <class 'str'>
text='我的邻居是：周杰轮，最喜欢：唱歌' <class 'langchain_core.prompt_values.StringPromptValue'>
参考下面示例：

问题：1+1等于几？，答案：2

问题：中国的首都是哪？，答案：北京

问题：太阳从哪边升起？，答案： <class 'str'>
messages=[SystemMessage(content='你是一个 helpful assistant。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你是谁？', additional_kwargs={}, response_metadata={})] <class 'langchain_core.prompt_values.ChatPromptValue'>


## 13. 13ChatPromptTemplate的使用

- 对应源码：`13ChatPromptTemplate的使用.py`
- 本节说明：演示 `ChatPromptTemplate + MessagesPlaceholder` 的标准组合，这是构建会话类应用的高频写法。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [15]:
if section_ready("13", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

    chat_prompt_template = ChatPromptTemplate.from_messages(
        [
            ("system", "你是一个边塞诗人，可以作诗。"),
            MessagesPlaceholder(variable_name="history"),
            ("human", "请再来一首唐诗。"),
        ]
    )

    history_data = [
        ("human", "你来写一个唐诗。"),
        ("ai", "床前明月光，疑是地上霜，举头望明月，低头思故乡。"),
        ("human", "好诗再来一个。"),
        ("ai", "锄禾日当午，汗滴禾下土，谁知盘中餐，粒粒皆辛苦。"),
    ]

    prompt_value = chat_prompt_template.invoke({"history": history_data})
    print(prompt_value.to_string())

    model = ChatTongyi(model="qwen3-max")
    res = (chat_prompt_template | model).invoke({"history": history_data})

    print(res.content, type(res))


[运行] 13
System: 你是一个边塞诗人，可以作诗。
Human: 你来写一个唐诗。
AI: 床前明月光，疑是地上霜，举头望明月，低头思故乡。
Human: 好诗再来一个。
AI: 锄禾日当午，汗滴禾下土，谁知盘中餐，粒粒皆辛苦。
Human: 请再来一首唐诗。
好的，这是边塞诗人岑参风格的一首原创唐诗：

《碛西征人》

黄云压戍楼，白草连天秋。  
孤城当绝漠，万骑出龙湫。  
箭裂霜风急，旗翻瀚海愁。  
功名何足道，埋骨即封侯。

——此诗以苍凉笔调写边塞征战之苦，末句“埋骨即封侯”反用汉代“生入玉门关”之意，道尽将士悲壮与无奈。 <class 'langchain_core.messages.ai.AIMessage'>


## 14. 14Chain的基础使用

- 对应源码：`14Chain的基础使用.py`
- 本节说明：把 Prompt 与 Model 通过 LCEL 串起来，体验最基本的链式调用。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [16]:
if section_ready("14", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

    chat_prompt_template = ChatPromptTemplate.from_messages(
        [
            ("system", "你是一个边塞诗人，可以作诗。"),
            MessagesPlaceholder(variable_name="history"),
            ("human", "请再来一首唐诗。"),
        ]
    )

    history_data = [
        ("human", "你来写一个唐诗。"),
        ("ai", "床前明月光，疑是地上霜，举头望明月，低头思故乡。"),
        ("human", "好诗再来一个。"),
        ("ai", "锄禾日当午，汗滴禾下土，谁知盘中餐，粒粒皆辛苦。"),
    ]

    model = ChatTongyi(model="qwen3-max")
    chain = chat_prompt_template | model

    for chunk in chain.stream({"history": history_data}):
        print(chunk.content, end="", flush=True)


[运行] 14
好的，这是我新作的一首边塞诗，请君品鉴：

**《塞上闻笛》**

黄云压戍楼，  
朔气卷寒流。  
孤城悬落日，  
万碛起边愁。  

夜静胡笳咽，  
风高战骨收。  
何须怨杨柳，  
明月照刀头。

——此诗以边塞苍凉为背景，融孤城、落日、胡笳、战骨等意象，末句“明月照刀头”既承古意，又见铁血豪情。不知可合君心？

## 15. 15[扩展]Python的或运算符的重写

- 对应源码：`15[扩展]Python的或运算符的重写.py`
- 本节说明：用纯 Python 理解 `|` 运算符重载，帮助从语言层面理解 LangChain 链式语法。
- 运行提示：直接可运行，无需额外前置条件。


In [17]:
class Test:
    def __init__(self, name):
        self.name = name

    def __or__(self, other):
        return MySequence(self, other)

    def __str__(self):
        return self.name


class MySequence:
    def __init__(self, *args):
        self.sequence = list(args)

    def __or__(self, other):
        self.sequence.append(other)
        return self

    def run(self):
        for item in self.sequence:
            print(item)


if __name__ == "__main__":
    a = Test("a")
    b = Test("b")
    c = Test("c")
    e = Test("e")
    f = Test("f")
    g = Test("g")

    d = a | b | c | e | f | g
    d.run()
    print(type(d))


a
b
c
e
f
g
<class '__main__.MySequence'>


## 16. 16Runnable接口源码查看

- 对应源码：`16Runnable接口源码查看.py`
- 本节说明：通过最小示例观察 `Runnable` 组合后的对象类型。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [18]:
if section_ready("16", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.llms import Tongyi
    from langchain_core.prompts import PromptTemplate

    prompt = PromptTemplate.from_template("请用一句话介绍 {topic}")
    model = Tongyi(model="qwen-max")

    chain = prompt | model

    print(type(chain))
    print(chain.invoke({"topic": "LangChain"}))


[运行] 16
<class 'langchain_core.runnables.base.RunnableSequence'>
LangChain是一个用于开发由语言模型驱动的应用程序的框架，它简化了链式思考、数据接入和交互等过程。


## 17. 17StrOutputParser解析器

- 对应源码：`17StrOutputParser解析器.py`
- 本节说明：把聊天模型输出直接解析成字符串，是最常见的输出解析器之一。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [19]:
if section_ready("17", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate

    parser = StrOutputParser()
    model = ChatTongyi(model="qwen3-max")

    prompt = ChatPromptTemplate.from_messages(
        [
            ("human", "我邻居姓：{lastname}，刚生了{gender}，请起名，仅告知我名字无需其它内容。"),
        ]
    )

    chain = prompt | model | parser

    res = chain.invoke({"lastname": "张", "gender": "女儿"})
    print(res)
    print(type(res))


[运行] 17
张若曦
<class 'langchain_core.messages.base.TextAccessor'>


## 18. 18JsonOutputParser解析器

- 对应源码：`18JsonOutputParser解析器.py`
- 本节说明：演示如何让模型按 JSON 格式返回，并把结构化结果继续送入下游链路。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [21]:
if section_ready("18", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from pydantic import BaseModel, Field
    from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
    from langchain_core.prompts import PromptTemplate
    from langchain_core.runnables import RunnableLambda
    from langchain_community.chat_models.tongyi import ChatTongyi


    class NamePayload(BaseModel):
        name: str = Field(description="只返回 1 个中文名字")


    str_parser = StrOutputParser()
    json_parser = JsonOutputParser(pydantic_object=NamePayload)
    model = ChatTongyi(model="qwen3-max")

    first_prompt = PromptTemplate.from_template(
        "我邻居姓：{lastname}，刚生了{gender}，请帮忙起名字，"
        "并封装为 JSON 格式返回给我。"
        "{format_instructions}"
    ).partial(format_instructions=json_parser.get_format_instructions())

    second_prompt = PromptTemplate.from_template(
        "姓名：{name}，请帮我解析含义。"
    )

    def normalize_name(payload: dict) -> dict:
        if "name" in payload and payload["name"]:
            return {"name": payload["name"]}
        if "name_suggestions" in payload and payload["name_suggestions"]:
            return {"name": payload["name_suggestions"][0]}
        raise ValueError(f"未从模型输出中解析到 name 字段，实际返回：{payload}")

    chain = (
        first_prompt
        | model
        | json_parser
        | RunnableLambda(normalize_name)
        | second_prompt
        | model
        | str_parser
    )

    for chunk in chain.stream({"lastname": "张", "gender": "女儿"}):
        print(chunk, end="", flush=True)



[运行] 18
“张婉清”是一个典型的中文姓名，我们可以从姓氏和名字两个部分来解析其含义：

---

### 一、姓氏：张（Zhāng）
- “张”是中国最常见的姓氏之一，位列《百家姓》前列。
- 本义为“拉开弓弦”，引申有“展开、扩大、张扬”之意，象征着积极进取、开放包容的精神。

---

### 二、名字：婉清（Wǎn Qīng）

#### 1. 婉（Wǎn）
- **字义**：温柔、和顺、美好。常用于形容女子温婉贤淑、言辞柔和。
- **文化意象**：在古典诗词中，“婉”常与“约”连用（如“婉约”），代表一种含蓄柔美的风格，也体现传统对女性优雅气质的赞美。
- **引申义**：委婉、婉转，有聪慧而不张扬的意味。

#### 2. 清（Qīng）
- **字义**：清澈、洁净、明朗、高洁。既可指自然之清（如清水、清风），也可指品格之清（如清廉、清雅）。
- **文化意象**：在中国传统文化中，“清”是备受推崇的品德，如“清正廉洁”“冰清玉洁”，象征纯洁无瑕、心境澄明。
- **音韵美感**：“清”字发音清亮悦耳，给人以清新脱俗之感。

---

### 三、整体寓意
“婉清”二字结合，既有**温婉柔美**的气质，又蕴含**清雅高洁**的品格。整个名字传递出一种：
- **外柔内刚**的女性形象；
- **秀外慧中、品行端正**的美好期许；
- 同时音韵和谐（wǎn qīng 平仄相协），读来婉转流畅，富有诗意。

---

### 四、文化联想
- 类似意境的名字常见于古典文学，如“婉如清扬”（出自《诗经·郑风·野有蔓草》：“有美一人，清扬婉兮”），正是对美丽、清雅女子的赞美。
- “婉清”也可理解为“温婉而清雅”，契合传统审美中对理想女性气质的描绘。

---

综上，“张婉清”是一个兼具**音韵美、字形美与内涵美**的名字，寄托了家人对其温柔贤淑、清雅高洁的美好祝愿。

## 19. 19RunnableLambda的基础使用

- 对应源码：`19RunnableLambda的基础使用.py`
- 本节说明：利用 `RunnableLambda` 在链路中做轻量数据转换。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [22]:
if section_ready("19", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnableLambda

    model = ChatTongyi(model="qwen3-max")
    str_parser = StrOutputParser()

    first_prompt = ChatPromptTemplate.from_messages(
        [
            ("human", "我邻居姓：{lastname}，刚生了{gender}，请帮忙起名字，仅生成一个名字，不要额外信息。"),
        ]
    )

    second_prompt = ChatPromptTemplate.from_messages(
        [
            ("human", "姓名：{name}，请帮我解析含义。"),
        ]
    )

    extract_name = RunnableLambda(lambda ai_msg: {"name": ai_msg.content.strip()})

    chain = first_prompt | model | extract_name | second_prompt | model | str_parser

    for chunk in chain.stream({"lastname": "曹", "gender": "女孩"}):
        print(chunk, end="", flush=True)


[运行] 19
“曹若溪”是一个富有诗意和文化内涵的中文名字，我们可以从姓氏、名字的字义以及整体意境三个方面来解析其含义：

---

### 一、姓氏：“曹”
- “曹”是中国常见姓氏之一，历史悠久，源自周代的官职名或封地名。
- 在传统文化中，“曹”并无特别负面或正面的象征意义，主要作为家族标识。

---

### 二、名字解析

#### 1. “若”字
- **基本含义**：有“如同、好像、仿佛”之意，常用于比喻或表达柔美、温婉的状态。
- **引申义**：在古文中也表示“顺从”“柔顺”，如《道德经》中有“上善若水”。
- **名字中的作用**：常用于女性名字中，增添文雅、含蓄、灵动的气质。

#### 2. “溪”字
- **基本含义**：指山间的小河或水流，清澈、细长、潺潺流淌。
- **文化意象**：在中国古典诗词中，“溪”常象征纯净、宁静、自然之美，如“明月松间照，清泉石上流”。
- **寓意**：代表温柔、灵动、生生不息，也有远离尘嚣、内心澄澈之意。

---

### 三、整体意境与寓意

“若溪”合起来可以理解为：
- **“如溪水一般”**：形容人如溪水般清澈、柔和、灵动、恬静。
- **诗意联想**：让人联想到山涧清溪、流水潺潺的画面，充满自然之美与宁静致远的意境。
- **性格寄望**：父母可能希望孩子拥有温柔善良、心思澄明、品性高洁、从容不迫的品格。

---

### 四、音韵美感
- “曹若溪”（Cáo Ruò Xī）读音平仄协调（阳平—去声—阴平），音调起伏有致，朗朗上口。
- 三个字均为常用字但不过于泛滥，既有文化底蕴又不失现代感。

---

### 总结：
**“曹若溪”是一个典雅、柔美、富有诗意的名字，寓意如溪水般清澈灵动、温婉娴静，寄托了对女孩纯净心灵与美好气质的期许。**

如果你是这个名字的主人，可以说你拥有一个非常有韵味且寓意美好的名字！

## 20. 20临时会话记忆

- 对应源码：`20临时会话记忆.py`
- 本节说明：使用内存型会话历史，让模型能记住同一个 session 内的上下文。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [23]:
if section_ready("20", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_core.chat_history import InMemoryChatMessageHistory
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    from langchain_core.runnables.history import RunnableWithMessageHistory

    model = ChatTongyi(model="qwen3-max")

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "你需要结合历史对话回答用户的问题。"),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}"),
        ]
    )

    str_parser = StrOutputParser()


    def print_prompt(full_prompt):
        print("=" * 20, full_prompt.to_string(), "=" * 20)
        return full_prompt


    base_chain = prompt | print_prompt | model | str_parser

    store = {}


    def get_history(session_id: str):
        if session_id not in store:
            store[session_id] = InMemoryChatMessageHistory()
        return store[session_id]


    conversation_chain = RunnableWithMessageHistory(
        base_chain,
        get_history,
        input_messages_key="input",
        history_messages_key="chat_history",
    )


    if __name__ == "__main__":
        session_config = {
            "configurable": {
                "session_id": "user_001",
            }
        }

        print(conversation_chain.invoke({"input": "小明有2只猫。"}, config=session_config))
        print(conversation_chain.invoke({"input": "小刚有1只狗。"}, config=session_config))
        print(conversation_chain.invoke({"input": "总共有几个宠物？"}, config=session_config))


[运行] 20
==================== System: 你需要结合历史对话回答用户的问题。
Human: 小明有2只猫。 ====================
好的，小明有2只猫。请问你想了解什么？比如它们的名字、品种，还是和小明之间的故事？或者这是一个数学问题的开头？欢迎继续补充！
==================== System: 你需要结合历史对话回答用户的问题。
Human: 小明有2只猫。
AI: 好的，小明有2只猫。请问你想了解什么？比如它们的名字、品种，还是和小明之间的故事？或者这是一个数学问题的开头？欢迎继续补充！
Human: 小刚有1只狗。 ====================
明白了！现在我们知道：

- 小明有 2 只猫，  
- 小刚有 1 只狗。

你是想比较他们养的宠物数量，还是想继续编故事、做数学题，或者有其他问题？欢迎告诉我下一步！ 😊
==================== System: 你需要结合历史对话回答用户的问题。
Human: 小明有2只猫。
AI: 好的，小明有2只猫。请问你想了解什么？比如它们的名字、品种，还是和小明之间的故事？或者这是一个数学问题的开头？欢迎继续补充！
Human: 小刚有1只狗。
AI: 明白了！现在我们知道：

- 小明有 2 只猫，  
- 小刚有 1 只狗。

你是想比较他们养的宠物数量，还是想继续编故事、做数学题，或者有其他问题？欢迎告诉我下一步！ 😊
Human: 总共有几个宠物？ ====================
小明有 2 只猫，小刚有 1 只狗，所以他们一共有：

**2 + 1 = 3 只宠物。**

答案是：**3 只宠物**。😺🐶


## 21. 21长期会话记忆

- 对应源码：`21长期会话记忆.py`
- 本节说明：把历史消息落盘到文件，实现跨进程的会话持久化。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [24]:
if section_ready("21", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    import json
    import os
    from typing import Sequence

    from langchain_community.chat_models import ChatTongyi
    from langchain_core.chat_history import BaseChatMessageHistory
    from langchain_core.messages import BaseMessage, message_to_dict, messages_from_dict
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    from langchain_core.runnables.history import RunnableWithMessageHistory


    class FileChatMessageHistory(BaseChatMessageHistory):
        def __init__(self, session_id: str, storage_path: str):
            self.session_id = session_id
            self.storage_path = storage_path
            self.file_path = os.path.join(self.storage_path, f"{self.session_id}.json")
            os.makedirs(self.storage_path, exist_ok=True)

        def add_messages(self, messages: Sequence[BaseMessage]) -> None:
            all_messages = list(self.messages)
            all_messages.extend(messages)

            payload = [message_to_dict(message) for message in all_messages]
            with open(self.file_path, "w", encoding="utf-8") as f:
                json.dump(payload, f, ensure_ascii=False, indent=2)

        @property
        def messages(self) -> list[BaseMessage]:
            try:
                with open(self.file_path, "r", encoding="utf-8") as f:
                    messages_data = json.load(f)
                return messages_from_dict(messages_data)
            except FileNotFoundError:
                return []

        def clear(self) -> None:
            with open(self.file_path, "w", encoding="utf-8") as f:
                json.dump([], f, ensure_ascii=False)


    model = ChatTongyi(model="qwen3-max")

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "你需要结合历史对话回答用户的问题。"),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}"),
        ]
    )

    str_parser = StrOutputParser()


    def print_prompt(full_prompt):
        print("=" * 20, full_prompt.to_string(), "=" * 20)
        return full_prompt


    base_chain = prompt | print_prompt | model | str_parser


    def get_history(session_id: str):
        return FileChatMessageHistory(session_id, "./chat_history")


    conversation_chain = RunnableWithMessageHistory(
        base_chain,
        get_history,
        input_messages_key="input",
        history_messages_key="chat_history",
    )


    if __name__ == "__main__":
        session_config = {
            "configurable": {
                "session_id": "user_001",
            }
        }

        print(conversation_chain.invoke({"input": "小明有2只猫。"}, config=session_config))
        print(conversation_chain.invoke({"input": "小刚有1只狗。"}, config=session_config))
        print(conversation_chain.invoke({"input": "总共有几个宠物？"}, config=session_config))


[运行] 21
==================== System: 你需要结合历史对话回答用户的问题。
Human: 小明有2只猫。 ====================
好的，小明有2只猫。请问你想了解什么？比如它们的名字、品种，还是和小明之间的故事？或者这是一个数学问题的开头？欢迎继续补充！
==================== System: 你需要结合历史对话回答用户的问题。
Human: 小明有2只猫。
AI: 好的，小明有2只猫。请问你想了解什么？比如它们的名字、品种，还是和小明之间的故事？或者这是一个数学问题的开头？欢迎继续补充！
Human: 小刚有1只狗。 ====================
明白了！现在我们知道：

- 小明有 2 只猫，  
- 小刚有 1 只狗。

如果你是在讲述一个故事，或者准备提出一个问题（比如关于宠物总数、比较数量、分配食物等），可以继续告诉我更多细节，我会帮你解答！
==================== System: 你需要结合历史对话回答用户的问题。
Human: 小明有2只猫。
AI: 好的，小明有2只猫。请问你想了解什么？比如它们的名字、品种，还是和小明之间的故事？或者这是一个数学问题的开头？欢迎继续补充！
Human: 小刚有1只狗。
AI: 明白了！现在我们知道：

- 小明有 2 只猫，  
- 小刚有 1 只狗。

如果你是在讲述一个故事，或者准备提出一个问题（比如关于宠物总数、比较数量、分配食物等），可以继续告诉我更多细节，我会帮你解答！
Human: 总共有几个宠物？ ====================
小明有 2 只猫，小刚有 1 只狗，所以他们一共有：

**2 + 1 = 3 只宠物。**

答案是：**3 只宠物**。


## 22. 22CSVLoader的使用

- 对应源码：`22CSVLoader的使用.py`
- 本节说明：从 CSV 文件加载文档对象，熟悉 `Document` 的结构。
- 运行提示：直接可运行，无需额外前置条件。


In [25]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path=str(DATA_DIR / "stu.csv"),
    csv_args={
        "delimiter": ",",
        "quotechar": '"',
    },
    encoding="utf-8",
)

for document in loader.lazy_load():
    print(document)


page_content='name: 王梓涵
age: 25
gender: 男
hobby: 吃饭,rap' metadata={'source': 'd:\\Agent\\03-RAG\\class\\代码\\AI大模型RAG与智能体开发\\P3_LangChainRAG开发\\data\\stu.csv', 'row': 0}
page_content='name: 刘若曦
age: 22
gender: 女
hobby: 睡觉,rap' metadata={'source': 'd:\\Agent\\03-RAG\\class\\代码\\AI大模型RAG与智能体开发\\P3_LangChainRAG开发\\data\\stu.csv', 'row': 1}
page_content='name: 陈俊宇
age: 20
gender: 男
hobby: 吃饭,rap' metadata={'source': 'd:\\Agent\\03-RAG\\class\\代码\\AI大模型RAG与智能体开发\\P3_LangChainRAG开发\\data\\stu.csv', 'row': 2}
page_content='name: 赵思瑶
age: 28
gender: 女
hobby: 睡觉,rap' metadata={'source': 'd:\\Agent\\03-RAG\\class\\代码\\AI大模型RAG与智能体开发\\P3_LangChainRAG开发\\data\\stu.csv', 'row': 3}
page_content='name: 黄浩然
age: 15
gender: 男
hobby: 吃饭,rap' metadata={'source': 'd:\\Agent\\03-RAG\\class\\代码\\AI大模型RAG与智能体开发\\P3_LangChainRAG开发\\data\\stu.csv', 'row': 4}
page_content='name: 林雨桐
age: 20
gender: 女
hobby: 唱跳,rap' metadata={'source': 'd:\\Agent\\03-RAG\\class\\代码\\AI大模型RAG与智能体开发\\P3_LangChainRAG开发\\data\\stu.cs

## 23. 23JSONLoader的使用

- 对应源码：`23JSONLoader的使用.py`
- 本节说明：从 JSON Lines 文件加载文档，需要 `jq` 包支持。
- 运行提示：运行前置：当前环境没有安装 jq。若条件不满足，本节会自动跳过。


In [26]:
if section_ready("23", HAS_JQ, "当前环境没有安装 jq"):
    from langchain_community.document_loaders import JSONLoader

    loader = JSONLoader(
        file_path=str(DATA_DIR / "stu_json_lines.json"),
        jq_schema=".",
        text_content=False,
        json_lines=True,
    )

    documents = loader.load()
    for document in documents:
        print(document)


[运行] 23
page_content='{"name": "\u5468\u6770\u8f6e", "age": 11, "gender": "\u7537"}' metadata={'source': 'D:\\Agent\\03-RAG\\class\\代码\\AI大模型RAG与智能体开发\\P3_LangChainRAG开发\\data\\stu_json_lines.json', 'seq_num': 1}
page_content='{"name": "\u8521\u4f9d\u4e34", "age": 12, "gender": "\u5973"}' metadata={'source': 'D:\\Agent\\03-RAG\\class\\代码\\AI大模型RAG与智能体开发\\P3_LangChainRAG开发\\data\\stu_json_lines.json', 'seq_num': 2}
page_content='{"name": "\u738b\u529b\u9e3f", "age": 11, "gender": "\u7537"}' metadata={'source': 'D:\\Agent\\03-RAG\\class\\代码\\AI大模型RAG与智能体开发\\P3_LangChainRAG开发\\data\\stu_json_lines.json', 'seq_num': 3}


## 24. 24PyPDFLoader的使用

- 对应源码：`24PyPDFLoader的使用.py`
- 本节说明：从 PDF 中提取文本，体验受密码保护 PDF 的读取方式。
- 运行提示：运行前置：当前环境没有安装 pypdf。若条件不满足，本节会自动跳过。


In [27]:
if section_ready("24", HAS_PYPDF, "当前环境没有安装 pypdf"):
    from langchain_community.document_loaders import PyPDFLoader

    loader = PyPDFLoader(
        file_path=str(DATA_DIR / "pdf2.pdf"),
        mode="single",
        password="itheima",
    )

    documents = loader.load()
    for index, doc in enumerate(documents, start=1):
        print(doc)
        print("=" * 20, index)


[运行] 24
page_content='天天开心，健康快乐。' metadata={'producer': '', 'creator': 'WPS 文字', 'creationdate': '2026-01-13T18:35:32+08:00', 'author': '曹宇[囧]', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-01-13T18:36:36+08:00', 'sourcemodified': "D:20260113183532+08'00'", 'subject': '', 'title': '', 'trapped': '/False', 'source': 'd:\\Agent\\03-RAG\\class\\代码\\AI大模型RAG与智能体开发\\P3_LangChainRAG开发\\data\\pdf2.pdf', 'total_pages': 1}
==================== 1


## 25. 25TextLoader和文档分割器

- 对应源码：`25TextLoader和文档分割器.py`
- 本节说明：先加载长文本，再切分成适合向量化的片段。
- 运行提示：直接可运行，无需额外前置条件。


In [28]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader(str(DATA_DIR / "Python基础语法.txt"), encoding="utf-8")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "！", "？", ".", "!", "?", " ", ""],
    length_function=len,
)

split_docs = splitter.split_documents(docs)
print(len(split_docs))

for doc in split_docs:
    print("=" * 20)
    print(doc)
    print("=" * 20)


56
page_content='## 一、Python 环境与程序运行
1.  Python 解释器：常用 CPython（官方默认），可通过命令行输入 python（Windows）/ python3（Linux/Mac）进入交互环境
2.  程序运行方式：
    - 交互环境：直接输入代码回车执行，适合简单测试
    - 脚本文件：将代码写入 .py 后缀文件，命令行执行 python 文件名.py（Windows）/ python3 文件名.py（Linux/Mac）
3.  注释：代码中不被执行的说明性文字，用于提高可读性
    - 单行注释：以 # 开头，示例：# 这是一行单行注释
    - 多行注释：用三个单引号 ''' 或三个双引号 """ 包裹，示例：
'''
这是多行注释
第一行
第二行
'''
"""
这也是多行注释
支持换行书写
"""' metadata={'source': 'd:\\Agent\\03-RAG\\class\\代码\\AI大模型RAG与智能体开发\\P3_LangChainRAG开发\\data\\Python基础语法.txt'}
page_content='## 二、变量与数据类型
1.  变量定义：无需声明类型，直接赋值即可，变量名由字母、数字、下划线组成，不能以数字开头，区分大小写，不能使用Python关键字
    - 赋值语法：变量名 = 赋值内容，示例：name = "Python"，age = 25，score = 98.5
    - 多变量赋值：a, b, c = 1, 2, 3（一一对应）；a = b = c = 10（所有变量赋同一值）
2.  基本数据类型
    - 数字类型（Number）
      - 整数（int）：正整数、负整数、0，示例：10，-5，0
      - 浮点数（float）：带小数点的数字，示例：3.14，-0.99，10.0
      - 布尔值（bool）：只有两个值 True（真，等价于1）和 False（假，等价于0），示例：is_ok = True，is_pass = False
      - 复数（complex）：形如 a+bj，示例：num = 3+4j
    - 字符串类型（str）：用单引号、双引号或三引号包裹的字符序列' metad

## 26. 26内存向量存储

- 对应源码：`26内存向量存储.py`
- 本节说明：在内存里构建向量库，适合理解最简版的相似度检索流程。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [29]:
if section_ready("26", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.document_loaders import CSVLoader
    from langchain_community.embeddings import DashScopeEmbeddings
    from langchain_core.vectorstores import InMemoryVectorStore

    vector_store = InMemoryVectorStore(
        embedding=DashScopeEmbeddings(model="text-embedding-v4")
    )

    loader = CSVLoader(
        file_path="./data/info.csv",
        encoding="utf-8",
        source_column="source",
    )

    documents = loader.load()
    ids = [f"id{i}" for i in range(1, len(documents) + 1)]

    vector_store.add_documents(documents=documents, ids=ids)
    vector_store.delete(ids=["id1", "id2"])

    result = vector_store.similarity_search("瑞达法", k=3)
    print(result)


[运行] 26
[Document(id='id7', metadata={'source': '黑马程序员', 'row': 6}, page_content='source: 黑马程序员\ninfo: 努力带来成就，Python助力辉煌'), Document(id='id10', metadata={'source': '黑马程序员', 'row': 9}, page_content='source: 黑马程序员\ninfo: 如何快速减肥呢'), Document(id='id5', metadata={'source': '传智教育', 'row': 4}, page_content='source: 传智教育\ninfo: Python学起来很简单的')]


## 27. 27外部向量持久化存储

- 对应源码：`27外部向量持久化存储.py`
- 本节说明：体验 Chroma 持久化向量库存储与检索。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [31]:
if section_ready("27", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_chroma import Chroma
    from langchain_community.embeddings import DashScopeEmbeddings

    vector_store = Chroma(
        collection_name="test",
        embedding_function=DashScopeEmbeddings(model="text-embedding-v1"),
        persist_directory="./chroma_db",
    )

    result = vector_store.similarity_search(
        "Python是不是简单易学呀",
        k=3,
        filter={"source": "黑马程序员"},
    )

    print(result)


[运行] 27
[Document(id='id7', metadata={'source': '黑马程序员', 'row': 6}, page_content='source: 黑马程序员\ninfo: 努力带来成就，Python助力辉煌'), Document(id='id8', metadata={'source': '黑马程序员', 'row': 7}, page_content='source: 黑马程序员\ninfo: 学习Python的时候也要记得好好休息打打篮球'), Document(id='id6', metadata={'row': 5, 'source': '黑马程序员'}, page_content='source: 黑马程序员\ninfo: 学习Python键盘敲烂月薪过万')]


## 28. 28向量检索构建提示词

- 对应源码：`28向量检索构建提示词.py`
- 本节说明：把检索结果拼接进提示词，形成最朴素的 RAG 问答链。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [32]:
if section_ready("28", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_community.embeddings import DashScopeEmbeddings
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.vectorstores import InMemoryVectorStore


    def print_prompt(prompt):
        print(prompt.to_string())
        print("=" * 20)
        return prompt


    model = ChatTongyi(model="qwen3-max")
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "请优先依据我提供的参考资料，简洁且专业地回答用户问题。\n参考资料：\n{context}"),
            ("human", "用户提问：{input}"),
        ]
    )

    vector_store = InMemoryVectorStore(
        embedding=DashScopeEmbeddings(model="text-embedding-v1")
    )

    vector_store.add_texts(
        [
            "减肥就是要少吃多练。",
            "在减脂期间吃东西很重要，清淡少油、控制卡路里摄入并运动起来。",
            "跑步是很好的运动哦。",
        ]
    )

    input_text = "怎么减肥？"
    result = vector_store.similarity_search(input_text, k=2)
    reference_text = "\n".join(f"- {doc.page_content}" for doc in result)

    chain = prompt | print_prompt | model | StrOutputParser()

    res = chain.invoke({"input": input_text, "context": reference_text})
    print(res)


[运行] 28
System: 请优先依据我提供的参考资料，简洁且专业地回答用户问题。
参考资料：
- 减肥就是要少吃多练。
- 在减脂期间吃东西很重要，清淡少油、控制卡路里摄入并运动起来。
Human: 用户提问：怎么减肥？
减肥的关键在于“少吃多练”：控制卡路里摄入，选择清淡少油的食物，并结合规律运动。在减脂期间，合理饮食与坚持锻炼同样重要。


## 29. 29RunnablePassthrough的使用

- 对应源码：`29RunnablePassthrough的使用.py`
- 本节说明：用 `RunnablePassthrough` 把用户输入与检索结果并行组织起来，形成更标准的 RAG 写法。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [33]:
if section_ready("29", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_community.embeddings import DashScopeEmbeddings
    from langchain_core.documents import Document
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnablePassthrough
    from langchain_core.vectorstores import InMemoryVectorStore


    def print_prompt(prompt):
        print(prompt.to_string())
        print("=" * 20)
        return prompt


    model = ChatTongyi(model="qwen3-max")
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "请优先依据我提供的参考资料，简洁且专业地回答用户问题。\n参考资料：\n{context}"),
            ("human", "用户提问：{input}"),
        ]
    )

    vector_store = InMemoryVectorStore(
        embedding=DashScopeEmbeddings(model="text-embedding-v1")
    )

    vector_store.add_texts(
        [
            "减肥就是要少吃多练。",
            "在减脂期间吃东西很重要，清淡少油、控制卡路里摄入并运动起来。",
            "跑步是很好的运动哦。",
        ]
    )

    retriever = vector_store.as_retriever(search_kwargs={"k": 2})


    def format_docs(docs: list[Document]) -> str:
        if not docs:
            return "无相关参考资料"

        return "\n".join(f"- {doc.page_content}" for doc in docs)


    chain = (
        {
            "input": RunnablePassthrough(),
            "context": retriever | format_docs,
        }
        | prompt
        | print_prompt
        | model
        | StrOutputParser()
    )

    res = chain.invoke("怎么减肥？")
    print(res)


[运行] 29
System: 请优先依据我提供的参考资料，简洁且专业地回答用户问题。
参考资料：
- 减肥就是要少吃多练。
- 在减脂期间吃东西很重要，清淡少油、控制卡路里摄入并运动起来。
Human: 用户提问：怎么减肥？
减肥的关键在于“少吃多练”：  
1. **饮食方面**：选择清淡少油的食物，严格控制每日卡路里摄入；  
2. **运动方面**：坚持规律锻炼，增加热量消耗。  

两者结合，才能有效减脂。
